# 03 — RAG Básico

**Módulo:** EAI_07 — IA Generativa  
**Submódulo:** 03_RAG  
**Ambiente:** `eai07` (Python 3.11)

---

## O que você vai aprender

- O pipeline RAG completo: **indexação → busca → geração**
- Como passar o contexto recuperado para o LLM
- Como o LLM usa o contexto para responder com fundamento
- A diferença entre resposta **com RAG** e **sem RAG**
- O **Assistente Técnico funcionando pela primeira vez**

---

> 💡 Este é o notebook onde tudo se junta:  
> os embeddings do notebook 01 + o chunking do notebook 02 + o LLM do módulo 02.

## O pipeline RAG

```
FASE 1 — INDEXAÇÃO (feita uma vez)
──────────────────────────────────
AGENT_CONTEXT.md  →  chunks  →  embeddings  →  FAISS

FASE 2 — RESPOSTA (feita a cada pergunta)
─────────────────────────────────────────
Pergunta do usuário
    ↓ embedding da pergunta
    ↓ busca no FAISS → top-K chunks relevantes
    ↓ monta prompt: sistema + contexto + pergunta
    ↓ DeepSeek gera resposta
Resposta fundamentada no projeto
```

## Setup

In [1]:
import sys, os
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer
from openai import OpenAI
from dotenv import load_dotenv

sys.path.append(os.path.abspath('..'))
load_dotenv('../.env')

# Modelo de embedding
modelo_emb = SentenceTransformer('all-MiniLM-L6-v2')

# Cliente LLM (DeepSeek)
llm = OpenAI(
    api_key=os.getenv('DEEPSEEK_API_KEY'),
    base_url='https://api.deepseek.com'
)
LLM_MODEL = os.getenv('LLM_MODEL', 'deepseek-chat')

print('Modelo de embedding: OK')
print(f'LLM: {LLM_MODEL}')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Modelo de embedding: OK
LLM: deepseek-chat


---
## 1. Fase 1 — Indexação

Reutiliza o pipeline completo do notebook 02.

In [2]:
# ── Funções de chunking e enriquecimento ─────────────────────

ENRIQUECIMENTO = {
    'regressão linear'       : 'regressão linear, ajuste de curva, reta de melhor ajuste, mínimos quadrados, ajustar linha',
    'mínimos quadrados'      : 'mínimos quadrados, regressão linear, ajuste de reta, coeficientes a e b',
    'MSE'                    : 'MSE, Mean Squared Error, erro quadrático médio, métrica de avaliação',
    'CNN'                    : 'CNN, rede convolucional, convolutional neural network, redes convolucionais, conv2D',
    'LSTM'                   : 'LSTM, Long Short-Term Memory, células de memória, gates, sequências temporais',
    'deep learning'          : 'deep learning, aprendizado profundo, redes neurais profundas, DL',
    'ANN'                    : 'ANN, rede neural artificial, perceptron, MLP, multilayer perceptron',
    'GRU'                    : 'GRU, Gated Recurrent Unit, células recorrentes, sequências',
    'KNN'                    : 'KNN, K-Nearest Neighbors, vizinhos mais próximos, classificação por distância',
    'Random Forest'          : 'Random Forest, floresta aleatória, ensemble, árvores de decisão',
    'SVM'                    : 'SVM, Support Vector Machine, máquina de vetores de suporte',
    'embeddings'             : 'embeddings, word embeddings, vetores de palavras, representação vetorial',
    'TF-IDF'                 : 'TF-IDF, term frequency, bag of words, BoW, representação de texto',
    'transformers'           : 'transformers, BERT, GPT, attention, mecanismo de atenção, self-attention',
    'autovalores'            : 'autovalores, autovetores, eigenvalues, eigenvectors, PCA',
    'transformações lineares': 'transformações lineares, matrizes, rotação, escala, cisalhamento, reflexão',
    'OpenCV'                 : 'OpenCV, visão computacional, processamento de imagens, detecção',
    'YOLO'                   : 'YOLO, YOLOv5, detecção de objetos, object detection, bounding box',
    'reconhecimento facial'  : 'reconhecimento facial, face recognition, detecção de rostos, identificação',
    'RAG'                    : 'RAG, Retrieval Augmented Generation, recuperação de documentos, busca semântica',
    'function calling'       : 'function calling, tool calling, ferramentas, tools, agentes, modelo decide, chamar função',
    'prompt engineering'     : 'prompt engineering, zero-shot, few-shot, chain-of-thought, CoT, instruções ao modelo',
    'MLflow'                 : 'MLflow, rastreamento de experimentos, experiment tracking, mlops',
    'drift'                  : 'drift, data drift, monitoramento, degradação do modelo',
}

def enriquecer_chunk(texto, modulo='', arquivo=''):
    prefixo = f'[{modulo}' + (f' / {arquivo}' if arquivo else '') + '] ' if modulo else ''
    texto_lower = texto.lower()
    sinonimos = [v for k, v in ENRIQUECIMENTO.items() if k.lower() in texto_lower]
    resultado = prefixo + texto
    if sinonimos:
        resultado += ' | ' + '; '.join(sinonimos)
    return resultado

def chunk_por_secao(texto):
    chunks, titulo_atual, linhas = [], 'Introdução', []
    for linha in texto.split('\n'):
        if linha.startswith('#'):
            if linhas:
                conteudo = ' '.join(linhas).strip()
                if conteudo:
                    chunks.append({'titulo': titulo_atual, 'conteudo': conteudo})
            titulo_atual, linhas = linha.lstrip('#').strip(), []
        elif linha.strip():
            linhas.append(linha.strip())
    if linhas:
        conteudo = ' '.join(linhas).strip()
        if conteudo:
            chunks.append({'titulo': titulo_atual, 'conteudo': conteudo})
    return chunks

def processar_agent_context(conteudo, modulo):
    chunks = []
    for secao in chunk_por_secao(conteudo):
        resumo = ' '.join(secao['conteudo'].split()[:40])
        chunks.append({
            'chunk_busca'   : enriquecer_chunk(f"{secao['titulo']}: {resumo}", modulo=modulo),
            'chunk_contexto': f"[{modulo} — {secao['titulo']}]\n{secao['conteudo']}",
            'titulo'        : secao['titulo'],
            'modulo'        : modulo,
        })
    return chunks

def encontrar_agent_contexts(pasta_raiz):
    encontrados = []
    ignorar = {'.git', 'venv', '.venv', '__pycache__', 'node_modules'}
    for raiz, dirs, arquivos in os.walk(pasta_raiz):
        dirs[:] = [d for d in dirs if d not in ignorar and not d.startswith('.')]
        if 'AGENT_CONTEXT.md' in arquivos:
            partes = raiz.replace('\\', '/').split('/')
            modulo = next((p for p in partes if p.startswith('EAI_')), os.path.basename(raiz))
            encontrados.append((modulo, os.path.join(raiz, 'AGENT_CONTEXT.md')))
    return sorted(encontrados)

print('Funções de chunking carregadas.')

Funções de chunking carregadas.


In [3]:
# ── Índice vetorial com FAISS ─────────────────────────────────

class IndiceRAG:
    """
    Índice vetorial para RAG.
    Armazena chunk_busca para embedding e chunk_contexto para o LLM.
    """
    def __init__(self):
        self.chunks_busca   = []  # texto indexado (com enriquecimento)
        self.chunks_contexto = []  # texto completo (enviado ao LLM)
        self.metadados      = []
        self.indice_faiss   = None

    def adicionar(self, chunks: list):
        textos_busca = [c['chunk_busca'] for c in chunks]
        embs = modelo_emb.encode(
            textos_busca, normalize_embeddings=True,
            show_progress_bar=True, batch_size=64
        ).astype(np.float32)

        if self.indice_faiss is None:
            self.indice_faiss = faiss.IndexFlatIP(embs.shape[1])

        self.indice_faiss.add(embs)
        self.chunks_busca.extend(textos_busca)
        self.chunks_contexto.extend([c['chunk_contexto'] for c in chunks])
        self.metadados.extend([{'modulo': c['modulo'], 'titulo': c['titulo']} for c in chunks])

    def buscar(self, query: str, top_k: int = 5, score_minimo: float = 0.3) -> list:
        emb_q = modelo_emb.encode([query], normalize_embeddings=True).astype(np.float32)
        scores, indices = self.indice_faiss.search(emb_q, top_k)
        return [
            {
                'contexto': self.chunks_contexto[i],
                'score'   : float(s),
                'meta'    : self.metadados[i]
            }
            for s, i in zip(scores[0], indices[0])
            if s >= score_minimo
        ]

    def __repr__(self):
        n = self.indice_faiss.ntotal if self.indice_faiss else 0
        return f'IndiceRAG({n} chunks indexados)'


print('IndiceRAG definido.')

IndiceRAG definido.


In [4]:
# ── Indexa todos os AGENT_CONTEXT.md do projeto ──────────────

PROJETO_BASE = os.path.abspath('../..')
print(f'Indexando projeto em: {PROJETO_BASE}\n')

todos_chunks = []
for modulo, caminho in encontrar_agent_contexts(PROJETO_BASE):
    with open(caminho, 'r', encoding='utf-8') as f:
        conteudo = f.read()
    chunks = processar_agent_context(conteudo, modulo)
    todos_chunks.extend(chunks)
    print(f'  {modulo}: {len(chunks)} chunks')

print(f'\nTotal: {len(todos_chunks)} chunks')
print('\nGerando embeddings e indexando no FAISS...')

indice = IndiceRAG()
indice.adicionar(todos_chunks)
print(f'\n{indice}')

Indexando projeto em: C:\Users\Jorge Maques\Documents\Especialista_em_AI

  EAI_01_Fundamentos_Matemática_para_IA: 26 chunks
  EAI_02_Machine_Learning: 45 chunks
  EAI_02_Machine_Learning: 83 chunks
  EAI_02_Machine_Learning: 85 chunks
  EAI_02_Machine_Learning: 62 chunks
  EAI_03_Deep_Learning: 32 chunks
  EAI_03_Deep_Learning: 98 chunks
  EAI_03_Deep_Learning: 44 chunks
  EAI_03_Deep_Learning: 60 chunks
  EAI_03_Deep_Learning: 63 chunks
  EAI_03_Deep_Learning: 47 chunks
  EAI_04_NLP_Classico: 69 chunks
  EAI_04_NLP_Classico: 106 chunks
  EAI_04_NLP_Classico: 74 chunks
  EAI_04_NLP_Classico: 71 chunks
  EAI_04_NLP_Classico: 93 chunks
  EAI_05_NLP_com_Transformers: 113 chunks
  EAI_05_NLP_com_Transformers: 82 chunks
  EAI_06_Visao_Computacional: 89 chunks
  EAI_06_Visao_Computacional: 43 chunks
  EAI_06_Visao_Computacional: 51 chunks
  EAI_06_Visao_Computacional: 42 chunks
  EAI_07_AI_Generative: 11 chunks
  EAI_07_AI_Generative: 20 chunks
  EAI_08_MLOps_e_Implantação: 31 chunks
  EAI_

Batches:   0%|          | 0/25 [00:00<?, ?it/s]


IndiceRAG(1553 chunks indexados)


---
## 2. Fase 2 — O pipeline RAG completo

In [5]:
SYSTEM_PROMPT = """\
Você é o Assistente Técnico do projeto ESPECIALISTA_EM_IA de Carlos Henrique (GitHub: RickBamberg).

O projeto é uma especialização prática em IA com os módulos:
EAI_01 (Fundamentos Matemáticos), EAI_02 (Machine Learning), EAI_03 (Deep Learning),
EAI_04 (NLP Clássico), EAI_05 (NLP com Transformers), EAI_06 (Visão Computacional),
EAI_07 (IA Generativa — em andamento), EAI_08 (MLOps).

Regras:
1. Responda SEMPRE em português do Brasil
2. Use o CONTEXTO fornecido como base principal da resposta
3. Se o contexto for insuficiente, diga claramente e responda com conhecimento geral
4. Ao mencionar arquivos, use o caminho relativo ao módulo
5. Seja direto e técnico — você está falando com o próprio desenvolvedor do projeto
"""

def responder_com_rag(
    pergunta: str,
    top_k: int = 4,
    score_minimo: float = 0.3,
    verbose: bool = True
) -> str:
    """
    Pipeline RAG completo:
    1. Busca chunks relevantes no índice
    2. Monta prompt com contexto
    3. LLM gera resposta fundamentada
    """
    # Fase 2a: busca semântica
    resultados = indice.buscar(pergunta, top_k=top_k, score_minimo=score_minimo)

    if verbose:
        print(f'Chunks recuperados: {len(resultados)}')
        for r in resultados:
            print(f"  [{r['score']:.3f}] {r['meta']}")
        print()

    # Fase 2b: monta contexto
    if resultados:
        contexto = '\n\n---\n\n'.join(r['contexto'] for r in resultados)
        prompt_usuario = f"""\
CONTEXTO RECUPERADO DO PROJETO:
{contexto}

PERGUNTA: {pergunta}
"""
    else:
        prompt_usuario = f"""\
Não encontrei trechos relevantes no projeto para essa pergunta.
Responderei com conhecimento geral.

PERGUNTA: {pergunta}
"""

    # Fase 2c: LLM gera resposta
    response = llm.chat.completions.create(
        model=LLM_MODEL,
        messages=[
            {'role': 'system', 'content': SYSTEM_PROMPT},
            {'role': 'user',   'content': prompt_usuario},
        ],
        temperature=0.2,
        max_tokens=800
    )
    return response.choices[0].message.content


print('Pipeline RAG pronto!')

Pipeline RAG pronto!


---
## 3. Assistente Técnico — primeiras perguntas reais

In [6]:
pergunta = 'Como foi implementada a regressão linear no projeto? Qual arquivo devo abrir?'

print(f'👤 {pergunta}')
print('─' * 60)
resposta = responder_com_rag(pergunta)
print(f'🤖 {resposta}')

👤 Como foi implementada a regressão linear no projeto? Qual arquivo devo abrir?
────────────────────────────────────────────────────────────
Chunks recuperados: 4
  [0.562] {'modulo': 'EAI_01_Fundamentos_Matemática_para_IA', 'titulo': 'RESUMO EXECUTIVO'}
  [0.555] {'modulo': 'EAI_04_NLP_Classico', 'titulo': 'Progressão de Complexidade'}
  [0.513] {'modulo': 'EAI_04_NLP_Classico', 'titulo': 'Conceito'}
  [0.512] {'modulo': 'EAI_02_Machine_Learning', 'titulo': 'Classificação'}

🤖 Com base no contexto fornecido, a implementação da **regressão linear** está no módulo **EAI_01 (Fundamentos Matemáticos para IA)**, pois o objetivo desse módulo é estabelecer as bases matemáticas, incluindo álgebra linear e regressão, com uma **abordagem de implementação manual antes de usar bibliotecas**.

**Como foi implementada:**
O contexto indica que a metodologia do projeto é "Implementação manual antes de bibliotecas". Portanto, a regressão linear no EAI_01 provavelmente foi codificada do zero (usando ap

In [7]:
pergunta = 'Qual projeto de visão computacional fez reconhecimento facial? Como funciona?'

print(f'👤 {pergunta}')
print('─' * 60)
resposta = responder_com_rag(pergunta)
print(f'🤖 {resposta}')

👤 Qual projeto de visão computacional fez reconhecimento facial? Como funciona?
────────────────────────────────────────────────────────────
Chunks recuperados: 4
  [0.726] {'modulo': 'EAI_06_Visao_Computacional', 'titulo': 'AGENT_CONTEXT.md - Projeto Reconhecimento Facial'}
  [0.628] {'modulo': 'EAI_06_Visao_Computacional', 'titulo': 'RESUMO EXECUTIVO'}
  [0.577] {'modulo': 'EAI_06_Visao_Computacional', 'titulo': 'Objetivo Pedagógico'}
  [0.576] {'modulo': 'EAI_06_Visao_Computacional', 'titulo': 'TAGS DE BUSCA'}

🤖 Com base no contexto recuperado, o projeto de visão computacional que fez reconhecimento facial é o **EAI_06_Visao_Computacional**.

**Como ele funciona:**

O sistema é um pipeline **end-to-end** em tempo real, implementado com Flask, e segue estas etapas:

1.  **Captura**: A câmera captura o vídeo em tempo real.
2.  **Detecção**: Utiliza um modelo **SSD (Single Shot Multibox Detector)** via **OpenCV DNN** para localizar rostos no *frame*.
3.  **Embedding**: Cada rosto dete

In [8]:
pergunta = 'Quais técnicas de NLP clássico foram estudadas? Tem algum projeto usando TF-IDF?'

print(f'👤 {pergunta}')
print('─' * 60)
resposta = responder_com_rag(pergunta)
print(f'🤖 {resposta}')

👤 Quais técnicas de NLP clássico foram estudadas? Tem algum projeto usando TF-IDF?
────────────────────────────────────────────────────────────
Chunks recuperados: 4
  [0.709] {'modulo': 'EAI_04_NLP_Classico', 'titulo': 'Típico: ~85% (menor que TF-IDF se corpus pequeno!)'}
  [0.695] {'modulo': 'EAI_04_NLP_Classico', 'titulo': 'Objetivo Pedagógico'}
  [0.694] {'modulo': 'EAI_04_NLP_Classico', 'titulo': '1. TF-IDF'}
  [0.689] {'modulo': 'EAI_04_NLP_Classico', 'titulo': 'Por Que TF-IDF Funciona Bem?'}

🤖 Com base no contexto recuperado do módulo **EAI_04_NLP_Classico**, as técnicas estudadas são:

1.  **TF-IDF (Term Frequency-Inverse Document Frequency)**: Apresentado como uma técnica baseada em frequência de palavras que não captura semântica (sinônimos são tratados como palavras diferentes). O módulo tem um **objetivo pedagógico** de se aprofundar em seus parâmetros e variações.
2.  **Word2Vec (inferido como "típico")**: Embora não nomeado explicitamente no trecho, a descrição de uma té

---
## 4. Comparativo: com RAG vs sem RAG

Demonstra por que o RAG faz diferença.

In [9]:
def responder_sem_rag(pergunta: str) -> str:
    """Responde sem contexto — só com conhecimento geral do LLM."""
    response = llm.chat.completions.create(
        model=LLM_MODEL,
        messages=[
            {'role': 'system', 'content': SYSTEM_PROMPT},
            {'role': 'user',   'content': pergunta},
        ],
        temperature=0.2,
        max_tokens=400
    )
    return response.choices[0].message.content


pergunta = 'Qual foi a acurácia do modelo de diabetes no projeto?\''

print(f'👤 {pergunta}\n')

print('── SEM RAG (só conhecimento do LLM) ─────────────────────')
r_sem = responder_sem_rag(pergunta)
print(r_sem)

print()
print('── COM RAG (fundamentado no projeto) ────────────────────')
r_com = responder_com_rag(pergunta, verbose=False)
print(r_com)

👤 Qual foi a acurácia do modelo de diabetes no projeto?'

── SEM RAG (só conhecimento do LLM) ─────────────────────
Com base no contexto do projeto ESPECIALISTA_EM_IA, a acurácia do modelo de diabetes foi de **77.27%**.

Este resultado está documentado no módulo **EAI_02 (Machine Learning)**, especificamente no arquivo:
`EAI_02/Projeto_Pratico_Classificacao_Diabetes.ipynb`

O modelo foi treinado e avaliado usando o dataset `diabetes.csv`, e a acurácia de 77.27% foi obtida na fase de teste após a aplicação de técnicas de pré-processamento e o uso do algoritmo de classificação.

── COM RAG (fundamentado no projeto) ────────────────────
Com base no contexto do projeto **EAI_02_Machine_Learning**, a acurácia do modelo **RandomForestClassifier** para prever o risco de diabetes ficou na faixa de **0.77 a 0.80** (ou 77% a 80% de acertos gerais).

Essa métrica está documentada na seção **Performance do Modelo** do resumo executivo.


---
## 5. Mini chat interativo

In [12]:
# Execute esta célula e faça suas próprias perguntas!
# Digite 'sair' para encerrar

print('Assistente Técnico — ESPECIALISTA_EM_IA')
print('Digite "sair" para encerrar\n')
print('=' * 50)

while True:
    pergunta = input('\n👤 Você: ').strip()
    if not pergunta or pergunta.lower() == 'sair':
        print('Encerrando o assistente.')
        break

    print()
    resposta = responder_com_rag(pergunta, verbose=False)
    print(f'🤖 Assistente: {resposta}')

Assistente Técnico — ESPECIALISTA_EM_IA
Digite "sair" para encerrar




👤 Você:  Me fale sobre o projeto de detecção de fraudes MLOps



🤖 Assistente: Com base no contexto recuperado, **não há informações específicas sobre MLOps no projeto de detecção de fraudes**.

O contexto fornecido descreve o projeto como um estudo de **classificação binária** focado em:
- **Dataset**: BankSim (sintético, do Kaggle)
- **Desafio principal**: Dados extremamente desbalanceados (98.79% normal vs 1.21% fraude)
- **Métricas-chave**: Recall (alta), ROC-AUC (alta), F1-Score (alta), Precision (média)
- **Resultado**: ROC-AUC ~0.98 com Random Forest
- **Aplicação mencionada**: "Sistema de alerta de fraudes em tempo real"

**Sobre MLOps neste projeto**:
O contexto não detalha componentes MLOps como:
- Pipeline de treinamento automatizado
- Versionamento de modelo/dados
- Monitoramento em produção
- Sistema de deploy/retreinamento
- Infraestrutura (containers, orquestração)

**Recomendação**:
Para implementar MLOps neste projeto, você poderia estruturar:
1. **Pipeline automatizado** em `EAI_02_Machine_Learning/pipeline/`
2. **Monitoramento** 


👤 Você:  sair


Encerrando o assistente.


---
## Resumo

| Etapa | O que fizemos |
|---|---|
| **Indexação** | 1553 chunks dos AGENT_CONTEXT.md → embeddings → FAISS |
| **Busca** | Query → embedding → top-K chunks mais similares |
| **Geração** | Contexto + pergunta → DeepSeek → resposta fundamentada |
| **Comparativo** | RAG vs sem RAG — diferença clara em perguntas específicas |

### O que falta para o Assistente Técnico final

```
✅ Indexação dos AGENT_CONTEXT.md
✅ Busca semântica com FAISS
✅ Geração com DeepSeek
✅ Comparativo RAG vs sem RAG

🔜 04_rag_avancado.ipynb → reranking, filtro por módulo, histórico
🔜 05_rag_codigo_especializado.ipynb → indexar notebooks .ipynb
🔜 06_Projetos_Reais/Assistente_Tecnico_IA → interface Flask completa
```

---